# Turkish MBT-C Emotion Labeling Pipeline (PZ / NG / EXCLUDE)

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"| VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: no GPU — voters will be extremely slow. Use an A100 runtime.")

In [ ]:
CONFIG = {
    # paths 
    "AUDIO_DIR":        "/content/drive/MyDrive/CHANGE_ME/segments",      # folder with the .wav files
    "TRANSCRIPTS_PATH": "/content/drive/MyDrive/CHANGE_ME/transcripts.csv", # .csv/.tsv/.json file OR folder of .txt files
    "OUTPUT_DIR":       "/content/drive/MyDrive/CHANGE_ME/labeling_outputs",

    # run mode
    "MODE": "pilot",        
    "PILOT_N": 300,
    "SEED": 42,

    # ensemble
    "VOTERS": {               # the 5-voter panel (+ reserve, off by default)
        "e2v": True,          # V1 emotion2vec+ large            (acoustic)
        "dim": True,          # V2 audeering valence/arousal     (acoustic)
        "whisper_ser": True,  # V3 Whisper-encoder SER           (acoustic)
        "text_llm": True,     # V4 transcript -> Qwen3-8B        (semantic, uses the manual)
        "audio_llm": True,    # V5 Qwen2.5-Omni-7B               (audio LLM, on probation)
    },
    "PRESET": "default",      # strict | default | lenient 

    # thresholds
    "THRESH": {
        "min_duration_s": 1.0,       
        "vad_min_speech_ratio": 0.30, # below this EXCLUDE(no_speech)
        "pole_mass_min": 0.40,        # V1/V3/R: min P(PZ)+P(NG) to vote
        "pole_margin_min": 0.15,      # V1/V3/R: min |P(PZ)-P(NG)| to vote
        "valence_pz": 0.60,           # V2: valence >= -> PZ
        "valence_ng": 0.40,           # V2: valence <= -> NG (between = abstain)
        "text_p_min": 0.50,           # V4: min prob of winning option to vote
        "audio_llm_tau": 0.50,        # V5: min |bias-corrected logprob gap| to vote
        "audio_llm_max_s": 20.0,      # V5: clips longer than this -> use LAST 20 s (Rule 2: terminal emotion)
    },
    "TERMINAL_WINDOW": False, # optional Rule-2 module for V1-V3 (re-score last 40% of clips > 8 s)

    # data details 
    "FILENAME_REGEX": r"^(?P<session>.+?)_SPEAKER_(?P<spk>\d+)_seg(?P<seg>\d+)$",
    "SPEAKER_MAP": {"SPEAKER_00": "therapist", "SPEAKER_01": "child"},
    "WHISPER_FALLBACK": True,     # transcribe clips that have no provided transcript
    "CONTEXT_NEIGHBORS": 1,       # +/- N neighboring transcripts as context for V4

    # infrastructure
    "CHECKPOINT_EVERY": 100,
    "HF_CACHE": "/content/hf_cache",
    "AUDIO_CACHE": "/content/audio16k",   # local 16 kHz mono cache
    "OFFLINE_AFTER_DOWNLOAD": True,       # block outbound calls once weights are cached
    "BATCH": {"vad": 1, "whisper": 8, "ser": 12, "text_llm": 1, "audio_llm": 1},
}

import os
os.environ["HF_HOME"] = CONFIG["HF_CACHE"]
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["DO_NOT_TRACK"] = "1"
os.makedirs(CONFIG["AUDIO_CACHE"], exist_ok=True)
VOTES_DIR = os.path.join(CONFIG["OUTPUT_DIR"], "votes")
print("Config loaded. MODE =", CONFIG["MODE"])

In [ ]:
%pip -q install "transformers>=4.56,<5" "accelerate>=1.0" "huggingface_hub>=0.34,<1.0" "funasr>=1.2.0" modelscope "silero-vad>=5.1" qwen-omni-utils soundfile librosa pandas scikit-learn tqdm
print("Installs done.")

In [ ]:
import os, re, gc, json, math, glob, random, warnings, traceback
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
from pathlib import Path
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float32
T = CONFIG["THRESH"]

os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
os.makedirs(VOTES_DIR, exist_ok=True)

def outpath(name): return os.path.join(CONFIG["OUTPUT_DIR"], name)

if CONFIG["OUTPUT_DIR"].startswith("/content/drive") and not os.path.ismount("/content/drive"):
    raise RuntimeError("Google Drive is not mounted but OUTPUT_DIR points into /content/drive. Run Cell 1.")

def free_gpu():
    """Call AFTER `del`-ing model variables in the cell — deleting here wouldn't drop the caller's reference."""
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def pending(voter_name, target_df):
    """Rows still to process for a voter — lets cells skip loading a model when there's nothing to do."""
    done = load_done_ids(os.path.join(VOTES_DIR, f"votes_{voter_name}.csv"))
    return target_df[~target_df["id"].isin(done)]

def load_done_ids(csv_path, retry_errors=True):
    """ids already processed; rows with status=='error' are retried on resume."""
    if not os.path.exists(csv_path): return set()
    try: df = pd.read_csv(csv_path, dtype={"id": str})
    except Exception: return set()
    if retry_errors and "status" in df.columns:
        df = df[df["status"] != "error"]
    return set(df["id"].astype(str))

def append_rows(csv_path, rows):
    if not rows: return
    df = pd.DataFrame(rows)
    df.to_csv(csv_path, mode="a", header=not os.path.exists(csv_path), index=False)

def read_results(csv_path):
    """Read a per-stage CSV keeping the LAST row per id (later rows supersede retried errors)."""
    if not os.path.exists(csv_path): return pd.DataFrame()
    df = pd.read_csv(csv_path, dtype={"id": str})
    return df.drop_duplicates(subset="id", keep="last")

# audio 
def cache_path(clip_id): return os.path.join(CONFIG["AUDIO_CACHE"], clip_id + ".wav")

def load_16k(path_or_id):
    """Return (float32 mono 16 kHz array, duration_s). Accepts a cache id or any path."""
    p = cache_path(path_or_id) if not str(path_or_id).endswith(".wav") else str(path_or_id)
    y, _ = librosa.load(p, sr=16000, mono=True)
    return y.astype(np.float32), len(y) / 16000.0

# filename parsing
FN_RE = re.compile(CONFIG["FILENAME_REGEX"])
def parse_filename(stem):
    m = FN_RE.match(stem)
    if not m: return {"session": None, "speaker_tag": None, "seg_idx": None}
    spk = f"SPEAKER_{m.group('spk')}"
    return {"session": m.group("session"), "speaker_tag": spk,
            "seg_idx": int(m.group("seg"))}

# vote mapping for categorical SER voters (V1 / V3 / R) 
POS_CLASSES = {"happy"}
NEG_CLASSES = {"angry", "sad", "fearful", "disgusted"}
ABSTAIN_CLASSES = {"neutral", "other", "unknown", "calm"}
SPLIT_CLASSES = {"surprised"}          # 50/50 
SYNONYMS = {"anger": "angry", "sadness": "sad", "fear": "fearful", "fearful": "fearful",
            "disgust": "disgusted", "happiness": "happy", "joy": "happy",
            "surprise": "surprised", "neutrality": "neutral"}

def categorical_to_vote(prob_dict):
    """prob_dict: {class_name: prob} -> (p_pz, p_ng, p_abstain, vote)"""
    p = {}
    for k, v in prob_dict.items():
        k = SYNONYMS.get(k.strip().lower(), k.strip().lower())
        p[k] = p.get(k, 0.0) + float(v)
    p_pz = sum(v for k, v in p.items() if k in POS_CLASSES) + 0.5 * sum(v for k, v in p.items() if k in SPLIT_CLASSES)
    p_ng = sum(v for k, v in p.items() if k in NEG_CLASSES) + 0.5 * sum(v for k, v in p.items() if k in SPLIT_CLASSES)
    p_ab = max(0.0, 1.0 - p_pz - p_ng)
    if (p_pz + p_ng) >= T["pole_mass_min"] and abs(p_pz - p_ng) >= T["pole_margin_min"]:
        vote = "PZ" if p_pz > p_ng else "NG"
    else:
        vote = "ABSTAIN"
    return round(p_pz, 4), round(p_ng, 4), round(p_ab, 4), vote

# generic voter runner 
def run_voter(voter_name, target_df, infer_fn, batch_size=1):
    """infer_fn(list_of_manifest_rows) -> list of dicts with p_pz/p_ng/p_abstain/vote/detail.
    Handles resume, checkpointing, and error rows."""
    csv_path = os.path.join(VOTES_DIR, f"votes_{voter_name}.csv")
    done = load_done_ids(csv_path)
    todo = target_df[~target_df["id"].isin(done)]
    print(f"[{voter_name}] {len(done)} done, {len(todo)} to process")
    buf = []
    rows = todo.to_dict("records")
    for i in tqdm(range(0, len(rows), batch_size), desc=voter_name):
        chunk = rows[i:i + batch_size]
        try:
            results = infer_fn(chunk)
        except Exception as e:
            results = [{"id": r["id"], "p_pz": "", "p_ng": "", "p_abstain": "", "vote": "ABSTAIN",
                        "detail": "", "status": "error", "error": repr(e)[:300]} for r in chunk]
        for r in results:
            r.setdefault("status", "ok"); r.setdefault("error", "")
        buf.extend(results)
        if len(buf) >= CONFIG["CHECKPOINT_EVERY"]:
            append_rows(csv_path, buf); buf = []
    append_rows(csv_path, buf)
    df = read_results(csv_path)
    n_err = int((df["status"] == "error").sum()) if "status" in df.columns else 0
    print(f"[{voter_name}] complete: {len(df)} rows ({n_err} errors — will retry on next run)")
    return df

def terminal_slice(y, sr=16000):
    """Rule 2 helper: last 40% of the clip."""
    n = len(y); return y[int(n * 0.6):]

print("Utilities ready. Device:", DEVICE)

In [ ]:
from huggingface_hub import snapshot_download

MODELS = {
    "e2v":         "emotion2vec/emotion2vec_plus_large",
    "sensevoice":  "FunAudioLLM/SenseVoiceSmall",
    "dim":         "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim",
    "whisper_ser": "firdhokk/speech-emotion-recognition-with-openai-whisper-large-v3",
    "xlsr_reserve":"ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition",
    "whisper":     "openai/whisper-large-v3",
    "text_llm":    "Qwen/Qwen3-8B",
    "audio_llm":   "Qwen/Qwen2.5-Omni-7B",
}
needed = {k for k, on in CONFIG["VOTERS"].items() if on and k not in CONFIG["DROP_VOTERS"]}
needed |= {"sensevoice"}
if CONFIG["WHISPER_FALLBACK"]: needed |= {"whisper"}

LOCAL = {}
for key in sorted(needed):
    repo = MODELS[key]
    print(f"downloading/caching {repo} ...")
    LOCAL[key] = snapshot_download(repo)  
print("All weights cached locally.")

if CONFIG["OFFLINE_AFTER_DOWNLOAD"]:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    print("HF offline mode ON — inference stages cannot make outbound calls.")

In [ ]:
from silero_vad import load_silero_vad, get_speech_timestamps

MANIFEST_CSV = outpath("manifest.csv")

def _fix_zip_name(info):
    """Zips made without the UTF-8 flag store names as cp437 — re-decode so Turkish chars survive."""
    if info.flag_bits & 0x800:
        return info.filename
    try: return info.filename.encode("cp437").decode("utf-8")
    except (UnicodeDecodeError, UnicodeEncodeError): return info.filename

if CONFIG["AUDIO_DIR"].lower().endswith(".zip"):
    import zipfile
    AUDIO_ROOT = "/content/segments_extracted"
    marker = os.path.join(AUDIO_ROOT, ".extract_done")
    if not os.path.exists(marker):
        print(f"Extracting {CONFIG['AUDIO_DIR']} to local disk (one-time per session)...")
        os.makedirs(AUDIO_ROOT, exist_ok=True)
        with zipfile.ZipFile(CONFIG["AUDIO_DIR"]) as zf:
            for info in tqdm(zf.infolist(), desc="unzip"):
                if info.is_dir(): continue
                name = _fix_zip_name(info)
                dest = os.path.join(AUDIO_ROOT, name.lstrip("/"))
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                with zf.open(info) as src, open(dest, "wb") as out:
                    out.write(src.read())
        open(marker, "w").write("ok")
    else:
        print("Zip already extracted this session.")
else:
    AUDIO_ROOT = CONFIG["AUDIO_DIR"]

wavs = sorted(glob.glob(os.path.join(AUDIO_ROOT, "**", "*.wav"), recursive=True))
if not wavs: raise RuntimeError(f"No .wav files under {AUDIO_ROOT}")
print(f"Found {len(wavs)} wav files.")

done = load_done_ids(MANIFEST_CSV)
vad_model = load_silero_vad()
buf = []
for p in tqdm(wavs, desc="manifest+vad"):
    stem = Path(p).stem
    if stem in done: continue
    row = {"id": stem, "path": p, **parse_filename(stem), "status": "ok", "error": ""}
    try:
        y, _ = librosa.load(p, sr=16000, mono=True)
        y = y.astype(np.float32)
        row["duration_s"] = round(len(y) / 16000.0, 3)
        cp = cache_path(stem)
        if not os.path.exists(cp): sf.write(cp, y, 16000, subtype="PCM_16")
        if row["duration_s"] >= T["min_duration_s"]:
            ts = get_speech_timestamps(torch.from_numpy(y), vad_model, sampling_rate=16000, return_seconds=True)
            speech = sum(t["end"] - t["start"] for t in ts)
            row["speech_ratio"] = round(speech / max(row["duration_s"], 1e-6), 3)
        else:
            row["speech_ratio"] = 0.0
    except Exception as e:
        row.update({"duration_s": 0.0, "speech_ratio": 0.0, "status": "error", "error": repr(e)[:300]})
    row["prefilter"] = ("io_error" if row["status"] == "error"
                        else "rule4_short" if row["duration_s"] < T["min_duration_s"]
                        else "no_speech" if row["speech_ratio"] < T["vad_min_speech_ratio"]
                        else "pass")
    buf.append(row)
    if len(buf) >= CONFIG["CHECKPOINT_EVERY"]: append_rows(MANIFEST_CSV, buf); buf = []
append_rows(MANIFEST_CSV, buf)

manifest = read_results(MANIFEST_CSV)
manifest["speaker_role"] = manifest["speaker_tag"].map(CONFIG["SPEAKER_MAP"]).fillna("unknown")

missing_cache = manifest[(manifest["prefilter"] == "pass") & (~manifest["id"].map(lambda i: os.path.exists(cache_path(i))))]
for r in tqdm(missing_cache.to_dict("records"), desc="rebuild cache"):
    try:
        y, _ = librosa.load(r["path"], sr=16000, mono=True)
        sf.write(cache_path(r["id"]), y.astype(np.float32), 16000, subtype="PCM_16")
    except Exception: pass

survivors = manifest[manifest["prefilter"] == "pass"].copy()
print(manifest["prefilter"].value_counts().to_string())
print(f"\nParsed session/seg metadata for {manifest['session'].notna().mean():.0%} of files.")
print(f"Speaker roles: {manifest['speaker_role'].value_counts().to_dict()}")

# pilot subset (stable across runs: seeded sample of survivors)
rng = random.Random(CONFIG["SEED"])
pilot_ids = set(rng.sample(sorted(survivors["id"]), min(CONFIG["PILOT_N"], len(survivors))))
manifest["pilot"] = manifest["id"].isin(pilot_ids)
survivors["pilot"] = survivors["id"].isin(pilot_ids)
TARGET = survivors[survivors["pilot"]] if CONFIG["MODE"] == "pilot" else survivors
print(f"\nMODE={CONFIG['MODE']} -> {len(TARGET)} clips targeted for the model stages.")

In [ ]:
# transcripts (provided plain-text + Whisper fallback + Rule-4 fillers) 
TRANSCRIPTS_CSV = outpath("transcripts.csv")

def load_provided_transcripts(path):
    """Accepts CSV/TSV (filename+text columns), JSON ({name: text} or list of dicts), or a folder of .txt."""
    mapping = {}
    def norm(k): return Path(str(k).strip()).stem
    if os.path.isdir(path):
        for t in glob.glob(os.path.join(path, "**", "*.txt"), recursive=True):
            mapping[norm(t)] = open(t, encoding="utf-8", errors="replace").read().strip()
    elif path.lower().endswith((".csv", ".tsv")):
        df = pd.read_csv(path, sep=None, engine="python")
        cols = list(df.columns)
        fcol = next((c for c in cols if re.search(r"file|name|id|wav|segment", str(c), re.I)), cols[0])
        tcol = next((c for c in cols if re.search(r"transcript|text|utterance", str(c), re.I)), cols[-1])
        for _, r in df.iterrows():
            mapping[norm(r[fcol])] = str(r[tcol]).strip()
    elif path.lower().endswith(".json"):
        data = json.load(open(path, encoding="utf-8"))
        if isinstance(data, dict):
            for k, v in data.items():
                mapping[norm(k)] = (v if isinstance(v, str) else str(v.get("text", v))).strip()
        else:
            for item in data:
                k = item.get("file") or item.get("filename") or item.get("id") or item.get("name")
                mapping[norm(k)] = str(item.get("transcript") or item.get("text") or "").strip()
    else:
        raise RuntimeError(f"Unsupported TRANSCRIPTS_PATH: {path}")
    return mapping

FILLER_TOKENS = {"mm","mmm","hm","hmm","hım","hıı","aa","aaa","a","ee","eee","e","ıı","ııı","ı",
                 "eh","ehh","ah","ahh","ıh","ıhı","hı","şey"}
def is_filler_only(text):
    if not isinstance(text, str): return True
    words = re.findall(r"[a-zçğıöşüâîû]+", text.lower())
    return len(words) == 0 or all(w in FILLER_TOKENS for w in words)

provided = load_provided_transcripts(CONFIG["TRANSCRIPTS_PATH"])
print(f"Provided transcripts loaded: {len(provided)}")

done = load_done_ids(TRANSCRIPTS_CSV)
todo = TARGET[~TARGET["id"].isin(done)]
matched = todo[todo["id"].map(lambda i: i in provided)]
unmatched = todo[~todo["id"].map(lambda i: i in provided)]
print(f"match rate on targeted clips: {len(matched)}/{len(todo)} "
      f"({(len(matched)/max(len(todo),1)):.0%}); {len(unmatched)} need Whisper fallback")

buf = [{"id": r["id"], "transcript": provided[r["id"]], "source": "provided",
        "filler_only": is_filler_only(provided[r["id"]]), "status": "ok", "error": ""}
       for r in matched.to_dict("records")]
append_rows(TRANSCRIPTS_CSV, buf)

if len(unmatched) and CONFIG["WHISPER_FALLBACK"]:
    from transformers import pipeline as hf_pipeline
    asr = hf_pipeline("automatic-speech-recognition", model=LOCAL["whisper"],
                      torch_dtype=DTYPE, device=DEVICE, batch_size=CONFIG["BATCH"]["whisper"])
    gk = {"language": "turkish", "task": "transcribe"}
    buf, rows = [], unmatched.to_dict("records")
    for i in tqdm(range(0, len(rows), CONFIG["BATCH"]["whisper"]), desc="whisper-fallback"):
        chunk = rows[i:i + CONFIG["BATCH"]["whisper"]]
        try:
            outs = asr([cache_path(r["id"]) for r in chunk], generate_kwargs=gk)
            for r, o in zip(chunk, outs):
                txt = (o["text"] or "").strip()
                buf.append({"id": r["id"], "transcript": txt, "source": "whisper",
                            "filler_only": is_filler_only(txt), "status": "ok", "error": ""})
        except Exception as e:
            buf.extend([{"id": r["id"], "transcript": "", "source": "whisper", "filler_only": True,
                         "status": "error", "error": repr(e)[:300]} for r in chunk])
        if len(buf) >= CONFIG["CHECKPOINT_EVERY"]: append_rows(TRANSCRIPTS_CSV, buf); buf = []
    append_rows(TRANSCRIPTS_CSV, buf)
    del asr; free_gpu()
elif len(unmatched):
    append_rows(TRANSCRIPTS_CSV, [{"id": r["id"], "transcript": "", "source": "missing",
                                   "filler_only": True, "status": "ok", "error": ""}
                                  for r in unmatched.to_dict("records")])

transcripts = read_results(TRANSCRIPTS_CSV)
print(transcripts["source"].value_counts().to_string())
print(f"filler-only/empty (Rule-4 candidates): {int(transcripts['filler_only'].sum())}")

In [ ]:
# SenseVoice audio events (laughter / crying) 
EVENTS_CSV = outpath("events.csv")
EVENT_TAGS = {"Laughter": "laughter", "Cry": "crying", "BGM": "bgm", "Applause": "applause",
              "Cough": "cough", "Sneeze": "sneeze", "Breath": "breath"}

done = load_done_ids(EVENTS_CSV)
todo = TARGET[~TARGET["id"].isin(done)]
print(f"[events] {len(done)} done, {len(todo)} to process")

if len(todo):
    from funasr import AutoModel as FunasrModel
    try:
        sv = FunasrModel(model=LOCAL["sensevoice"], trust_remote_code=True, disable_update=True, device=DEVICE)
    except Exception as e:
        print("local-path load failed, retrying via modelscope id:", repr(e)[:150])
        sv = FunasrModel(model="iic/SenseVoiceSmall", trust_remote_code=True, disable_update=True, device=DEVICE)

    def infer_events(chunk):
        out = []
        for r in chunk:
            res = sv.generate(input=cache_path(r["id"]), language="auto", use_itn=False)
            raw = res[0].get("text", "") if res else ""
            tags = set(re.findall(r"<\|([A-Za-z_]+)\|>", raw))
            events = sorted({EVENT_TAGS[t] for t in tags if t in EVENT_TAGS})
            out.append({"id": r["id"], "events": ";".join(events), "raw_tags": ";".join(sorted(tags))[:200],
                        "status": "ok", "error": ""})
        return out

    buf, rows = [], todo.to_dict("records")
    for i in tqdm(range(0, len(rows), 8), desc="sensevoice"):
        chunk = rows[i:i + 8]
        try: buf.extend(infer_events(chunk))
        except Exception as e:
            buf.extend([{"id": r["id"], "events": "", "raw_tags": "", "status": "error",
                         "error": repr(e)[:300]} for r in chunk])
        if len(buf) >= CONFIG["CHECKPOINT_EVERY"]: append_rows(EVENTS_CSV, buf); buf = []
    append_rows(EVENTS_CSV, buf)
    del sv; free_gpu()

events = read_results(EVENTS_CSV)
ev_counts = pd.Series(";".join(events["events"].fillna("").astype(str)).split(";")).value_counts()
print(ev_counts[ev_counts.index != ""].to_string() if len(ev_counts) else "no events detected")


## The voter panel

In [ ]:
# V1 — emotion2vec+ large (acoustic backbone) 
if CONFIG["VOTERS"]["e2v"] and "e2v" not in CONFIG["DROP_VOTERS"] and len(pending("e2v", TARGET)):
    from funasr import AutoModel as FunasrModel
    try:
        e2v = FunasrModel(model=LOCAL["e2v"], disable_update=True, device=DEVICE)
    except Exception as e:
        print("local-path load failed, retrying via modelscope id:", repr(e)[:150])
        e2v = FunasrModel(model="iic/emotion2vec_plus_large", disable_update=True, device=DEVICE)

    def _e2v_probs(y):
        res = e2v.generate(input=y, fs=16000, granularity="utterance", extract_embedding=False)
        labels = [l.split("/")[-1].strip() for l in res[0]["labels"]]
        return dict(zip(labels, [float(s) for s in res[0]["scores"]]))

    def infer_e2v(chunk):
        out = []
        for r in chunk:
            y, dur = load_16k(r["id"])
            probs = _e2v_probs(y)
            if CONFIG["TERMINAL_WINDOW"] and dur > 8.0:
                p2 = _e2v_probs(terminal_slice(y))
                v_full, v_term = categorical_to_vote(probs), categorical_to_vote(p2)
                if v_term[3] != "ABSTAIN" and v_term[3] != v_full[3] and abs(v_term[0]-v_term[1]) >= 1.5*T["pole_margin_min"]:
                    probs = p2   # Rule 2: terminal emotion wins for this voter
            p_pz, p_ng, p_ab, vote = categorical_to_vote(probs)
            out.append({"id": r["id"], "p_pz": p_pz, "p_ng": p_ng, "p_abstain": p_ab,
                        "vote": vote, "detail": json.dumps(probs, ensure_ascii=False)})
        return out

    votes_e2v = run_voter("e2v", TARGET, infer_e2v, batch_size=1)
    del e2v; free_gpu()
else:
    print("V1 e2v disabled/dropped/already complete.")

In [ ]:
# V2 — audeering dimensional valence (acoustic) 
if CONFIG["VOTERS"]["dim"] and "dim" not in CONFIG["DROP_VOTERS"] and len(pending("dim", TARGET)):
    import torch.nn as nn
    from transformers import Wav2Vec2Processor
    from transformers.models.wav2vec2.modeling_wav2vec2 import Wav2Vec2Model, Wav2Vec2PreTrainedModel

    class RegressionHead(nn.Module):        # from audeering/w2v2-how-to
        def __init__(self, config):
            super().__init__()
            self.dense = nn.Linear(config.hidden_size, config.hidden_size)
            self.dropout = nn.Dropout(config.final_dropout)
            self.out_proj = nn.Linear(config.hidden_size, config.num_labels)
        def forward(self, x):
            x = self.dropout(x); x = torch.tanh(self.dense(x)); x = self.dropout(x)
            return self.out_proj(x)

    class EmotionModel(Wav2Vec2PreTrainedModel):
        def __init__(self, config):
            super().__init__(config)
            self.wav2vec2 = Wav2Vec2Model(config)
            self.classifier = RegressionHead(config)
            self.init_weights()
        def forward(self, input_values):
            hidden = self.wav2vec2(input_values)[0].mean(dim=1)
            return self.classifier(hidden)   # [arousal, dominance, valence] in ~0..1

    dim_proc = Wav2Vec2Processor.from_pretrained(LOCAL["dim"])
    dim_model = EmotionModel.from_pretrained(LOCAL["dim"]).to(DEVICE).eval()

    @torch.no_grad()
    def _adv(y):
        iv = dim_proc(y, sampling_rate=16000, return_tensors="pt")["input_values"].to(DEVICE)
        a, d, v = dim_model(iv)[0].float().cpu().numpy().tolist()
        return a, d, v

    def infer_dim(chunk):
        out = []
        for r in chunk:
            y, dur = load_16k(r["id"])
            a, d, v = _adv(y)
            if CONFIG["TERMINAL_WINDOW"] and dur > 8.0:
                a2, d2, v2 = _adv(terminal_slice(y))
                if (v2 >= T["valence_pz"]) != (v >= T["valence_pz"]) and abs(v2 - 0.5) > abs(v - 0.5):
                    a, d, v = a2, d2, v2
            vote = "PZ" if v >= T["valence_pz"] else "NG" if v <= T["valence_ng"] else "ABSTAIN"
            span = max(T["valence_pz"] - T["valence_ng"], 1e-6)
            p_pz = float(np.clip((v - T["valence_ng"]) / span, 0, 1))
            out.append({"id": r["id"], "p_pz": round(p_pz, 4), "p_ng": round(1 - p_pz, 4),
                        "p_abstain": 0.0, "vote": vote,
                        "detail": json.dumps({"arousal": round(a,4), "dominance": round(d,4), "valence": round(v,4)})})
        return out

    votes_dim = run_voter("dim", TARGET, infer_dim, batch_size=1)
    del dim_model, dim_proc; free_gpu()
else:
    print("V2 dim disabled/dropped/already complete.")

In [ ]:
# V3 — Whisper-encoder SER (acoustic) 
if CONFIG["VOTERS"]["whisper_ser"] and "whisper_ser" not in CONFIG["DROP_VOTERS"] and len(pending("whisper_ser", TARGET)):
    from transformers import pipeline as hf_pipeline
    ser = hf_pipeline("audio-classification", model=LOCAL["whisper_ser"],
                      torch_dtype=torch.float32, device=DEVICE)

    def _ser_probs(y):
        res = ser({"array": y, "sampling_rate": 16000}, top_k=None)
        return {r["label"]: r["score"] for r in res}

    def infer_wser(chunk):
        out = []
        for r in chunk:
            y, dur = load_16k(r["id"])
            probs = _ser_probs(y)
            if CONFIG["TERMINAL_WINDOW"] and dur > 8.0:
                p2 = _ser_probs(terminal_slice(y))
                v_full, v_term = categorical_to_vote(probs), categorical_to_vote(p2)
                if v_term[3] != "ABSTAIN" and v_term[3] != v_full[3] and abs(v_term[0]-v_term[1]) >= 1.5*T["pole_margin_min"]:
                    probs = p2
            p_pz, p_ng, p_ab, vote = categorical_to_vote(probs)
            out.append({"id": r["id"], "p_pz": p_pz, "p_ng": p_ng, "p_abstain": p_ab,
                        "vote": vote, "detail": json.dumps({k: round(v,4) for k, v in probs.items()})})
        return out

    votes_wser = run_voter("whisper_ser", TARGET, infer_wser, batch_size=1)
    del ser; free_gpu()
else:
    print("V3 whisper_ser disabled/dropped/already complete.")

In [ ]:
# V4 — transcript → Qwen3-8B with the labeling manual (semantic) 
if CONFIG["VOTERS"]["text_llm"] and "text_llm" not in CONFIG["DROP_VOTERS"] and len(pending("text_llm", TARGET)):
    from transformers import AutoTokenizer, AutoModelForCausalLM

    MANUAL_SYSTEM_PROMPT = (
        "Manual"
    )

    tok = AutoTokenizer.from_pretrained(LOCAL["text_llm"])
    llm = AutoModelForCausalLM.from_pretrained(LOCAL["text_llm"], torch_dtype=DTYPE,
                                               device_map=DEVICE, attn_implementation="sdpa").eval()

    tmap = transcripts.set_index("id")
    emap = events.set_index("id") if len(events) else None
    # Rule-5 context: neighbors by (session, seg_idx)
    ctx = manifest[manifest["session"].notna()][["id", "session", "seg_idx"]].copy()
    ctx_lookup = {}
    for sess, g in ctx.groupby("session"):
        g = g.sort_values("seg_idx")
        ids = g["id"].tolist()
        for i, cid in enumerate(ids):
            lo, hi = max(0, i - CONFIG["CONTEXT_NEIGHBORS"]), min(len(ids), i + CONFIG["CONTEXT_NEIGHBORS"] + 1)
            ctx_lookup[cid] = [x for x in ids[lo:hi] if x != cid]

    def _get_txt(cid):
        try:
            v = tmap.loc[cid, "transcript"]
            return "" if pd.isna(v) else str(v)
        except Exception: return ""

    def build_user_msg(r, order):
        cid = r["id"]
        txt = _get_txt(cid)
        ev = ""
        if emap is not None and cid in emap.index:
            ev_raw = emap.loc[cid, "events"]
            ev = "" if pd.isna(ev_raw) else str(ev_raw)
        parts = []
        neigh = [n for n in ctx_lookup.get(cid, []) if _get_txt(n)]
        if neigh:
            parts.append("CONTEXT (neighboring segments): " + " | ".join(f"«{_get_txt(n)}»" for n in neigh))
        parts.append(f"TARGET segment (speaker: {r.get('speaker_role','unknown')}, {r.get('duration_s','?')} s): «{txt}»")
        parts.append("Detected acoustic events: " + (ev if ev else "none"))
        opts = "PZ, NG, or EXCLUDE" if order == 0 else "NG, PZ, or EXCLUDE"
        parts.append(f"Answer with exactly one token — {opts}:")
        return "\n".join(parts)

    @torch.no_grad()
    def score_options(prompt_text, options):
        prompt_ids = tok(prompt_text, add_special_tokens=False)["input_ids"]
        seqs, lens = [], []
        for o in options:
            cids = tok(o, add_special_tokens=False)["input_ids"]
            seqs.append(prompt_ids + cids); lens.append(len(cids))
        maxlen = max(len(s) for s in seqs)
        pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
        input_ids = torch.full((len(seqs), maxlen), pad, dtype=torch.long)
        attn = torch.zeros((len(seqs), maxlen), dtype=torch.long)
        for i, s in enumerate(seqs):
            input_ids[i, :len(s)] = torch.tensor(s); attn[i, :len(s)] = 1
        logits = llm(input_ids=input_ids.to(DEVICE), attention_mask=attn.to(DEVICE)).logits.float().log_softmax(-1)
        lps = []
        for i, s in enumerate(seqs):
            lp = sum(logits[i, pos - 1, s[pos]].item() for pos in range(len(s) - lens[i], len(s)))
            lps.append(lp)
        e = np.exp(np.array(lps) - max(lps)); return (e / e.sum()).tolist()

    OPTIONS = ["PZ", "NG", "EXCLUDE"]
    def infer_text(chunk):
        out = []
        for r in chunk:
            cid = r["id"]
            txt = _get_txt(cid)
            if is_filler_only(txt):   # unreliable/empty transcript -> a text voter must not vote
                out.append({"id": cid, "p_pz": 0.0, "p_ng": 0.0, "p_abstain": 1.0,
                            "vote": "ABSTAIN", "detail": "no_reliable_transcript"})
                continue
            probs = np.zeros(3)
            for order in (0, 1):
                msgs = [{"role": "system", "content": MANUAL_SYSTEM_PROMPT},
                        {"role": "user", "content": build_user_msg(r, order)}]
                prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                                 enable_thinking=False)
                probs += np.array(score_options(prompt, OPTIONS)) / 2.0
            p_pz, p_ng, p_ex = [round(float(x), 4) for x in probs]
            top = OPTIONS[int(np.argmax(probs))]
            vote = top if (top != "EXCLUDE" and probs.max() >= T["text_p_min"]) else "ABSTAIN"
            out.append({"id": cid, "p_pz": p_pz, "p_ng": p_ng, "p_abstain": p_ex,
                        "vote": vote, "detail": ""})
        return out

    votes_text = run_voter("text_llm", TARGET, infer_text, batch_size=CONFIG["BATCH"]["text_llm"])
    del llm, tok; free_gpu()
else:
    print("V4 text_llm disabled/dropped/already complete.")

In [ ]:
# V5 — Qwen2.5-Omni-7B forced-choice (audio LLM, on probation) 
if CONFIG["VOTERS"]["audio_llm"] and "audio_llm" not in CONFIG["DROP_VOTERS"] and len(pending("audio_llm", TARGET)):
    from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
    try:
        from qwen_omni_utils import process_mm_info
    except ImportError:
        process_mm_info = None

    try:
        omni = Qwen2_5OmniForConditionalGeneration.from_pretrained(
            LOCAL["audio_llm"], torch_dtype=DTYPE, device_map=DEVICE,
            attn_implementation="sdpa", enable_audio_output=False).eval()
    except TypeError:  # transformers version without the enable_audio_output kwarg
        omni = Qwen2_5OmniForConditionalGeneration.from_pretrained(
            LOCAL["audio_llm"], torch_dtype=DTYPE, device_map=DEVICE,
            attn_implementation="sdpa").eval()
    try: omni.disable_talker()
    except Exception: pass
    # The composite wrapper has no forward(); logits come from the thinker LM.
    OMNI_LM = getattr(omni, "thinker", omni)
    oproc = Qwen2_5OmniProcessor.from_pretrained(LOCAL["audio_llm"])

    def _omni_inputs(conv, y):
        """Official usage pattern (chat template + process_mm_info), with kwarg fallbacks."""
        text = oproc.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
        auds = [y]
        if process_mm_info is not None:
            try:
                auds, _imgs, _vids = process_mm_info(conv, use_audio_in_video=False)
            except Exception:
                auds = [y]
        try:
            return oproc(text=text, audio=auds, return_tensors="pt", padding=True)
        except TypeError:  # some versions name the kwarg `audios`
            return oproc(text=text, audios=auds, return_tensors="pt", padding=True)

    SYS = ("")
    QUESTIONS = ["Is the dominant emotion positive (PZ) or negative (NG)? Answer PZ or NG:",
                 "Is the dominant emotion negative (NG) or positive (PZ)? Answer NG or PZ:"]

    @torch.no_grad()
    def omni_gap(y):
        """Return mean logprob gap lp(PZ)-lp(NG) over 2 question orders x 2 candidates."""
        if len(y) > int(T["audio_llm_max_s"] * 16000):
            y = y[-int(T["audio_llm_max_s"] * 16000):]    
        gaps = []
        for q in QUESTIONS:
            conv = [{"role": "system", "content": [{"type": "text", "text": SYS}]},
                    {"role": "user", "content": [{"type": "audio", "audio": y},
                                                 {"type": "text", "text": q}]}]
            binputs = _omni_inputs(conv, y)
            lps = []
            for cand in ("PZ", "NG"):
                # append candidate token ids to the tokenized prompt (avoids retokenization
                # boundary effects between the chat template tail and the candidate string)
                cand_ids = oproc.tokenizer(cand, add_special_tokens=False)["input_ids"]
                inputs = dict(binputs)
                inputs["input_ids"] = torch.cat(
                    [binputs["input_ids"], torch.tensor([cand_ids], dtype=torch.long)], dim=1)
                inputs["attention_mask"] = torch.cat(
                    [binputs["attention_mask"], torch.ones((1, len(cand_ids)), dtype=torch.long)], dim=1)
                moved = {}
                for k, v in inputs.items():
                    if hasattr(v, "to"):
                        v = v.to(DEVICE)
                        if torch.is_floating_point(v): v = v.to(DTYPE)
                    moved[k] = v
                logits = OMNI_LM(**moved).logits.float().log_softmax(-1)
                ids = moved["input_ids"][0]
                lp = sum(logits[0, pos - 1, ids[pos]].item()
                         for pos in range(len(ids) - len(cand_ids), len(ids)))
                lps.append(lp)
            gaps.append(lps[0] - lps[1])
        return float(np.mean(gaps))

    def infer_omni(chunk):
        out = []
        for r in chunk:
            y, _ = load_16k(r["id"])
            gap = omni_gap(y)
            # raw gap stored; bias correction + tau gating happen in the aggregation cell
            p_pz = float(1.0 / (1.0 + math.exp(-gap)))
            out.append({"id": r["id"], "p_pz": round(p_pz, 4), "p_ng": round(1 - p_pz, 4),
                        "p_abstain": 0.0, "vote": "RAW", "detail": json.dumps({"gap": round(gap, 4)})})
        return out

    votes_omni = run_voter("audio_llm", TARGET, infer_omni, batch_size=CONFIG["BATCH"]["audio_llm"])
    del omni, oproc, OMNI_LM; free_gpu()
else:
    print("V5 audio_llm disabled/dropped/already complete.")

In [ ]:
# PILOT REPORT (before running MODE='full') 
from sklearn.metrics import cohen_kappa_score

ACTIVE = [v for v, on in CONFIG["VOTERS"].items() if on and v not in CONFIG["DROP_VOTERS"]]

def load_votes(name):
    df = read_results(os.path.join(VOTES_DIR, f"votes_{name}.csv"))
    if df.empty: return df
    df = df[df["status"] != "error"] if "status" in df.columns else df
    df = df[["id", "p_pz", "p_ng", "p_abstain", "vote", "detail"]].copy()
    for c in ("p_pz", "p_ng", "p_abstain"):   # error rows can leave "" behind -> force numeric
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def calibrated_omni_votes(df, pilot_only_ids):
    """Bias-correct the raw logprob gap using the pilot median, then gate by tau."""
    if df.empty: return df, 0.0
    gaps = df["detail"].map(lambda d: json.loads(d).get("gap", 0.0) if isinstance(d, str) and d.startswith("{") else 0.0)
    df = df.assign(gap=gaps)
    pilot_gaps = df[df["id"].isin(pilot_only_ids)]["gap"]
    bias = float(pilot_gaps.median()) if len(pilot_gaps) >= 30 else float(df["gap"].median())
    g = df["gap"] - bias
    df["p_pz"] = (1 / (1 + np.exp(-g))).round(4)
    df["p_ng"] = (1 - df["p_pz"]).round(4)
    df["vote"] = np.where(g >= T["audio_llm_tau"], "PZ", np.where(g <= -T["audio_llm_tau"], "NG", "ABSTAIN"))
    return df, bias

pilot_ids_set = set(manifest[manifest["pilot"]]["id"])
ALL_VOTES = {}
for v in ACTIVE:
    df = load_votes(v)
    if v == "audio_llm" and not df.empty:
        df, omni_bias = calibrated_omni_votes(df, pilot_ids_set)
        print(f"[audio_llm] pilot bias correction: {omni_bias:+.3f} logprob units")
    ALL_VOTES[v] = df

print("\n================ PILOT REPORT ================")
report_ids = pilot_ids_set
DEGENERATE = []
for v in ACTIVE:
    df = ALL_VOTES[v]
    df = df[df["id"].isin(report_ids)]
    if df.empty:
        print(f"\n{v}: NO DATA — did its cell run?"); continue
    dist = df["vote"].value_counts(normalize=True)
    pz, ng, ab = (dist.get("PZ", 0), dist.get("NG", 0), dist.get("ABSTAIN", 0))
    n_votes = int((df["vote"] != "ABSTAIN").sum())
    flag = ""
    if n_votes >= 20:
        pole_share = max(pz, ng) / max(pz + ng, 1e-9)
        if pole_share > 0.90: flag = " << DEGENERATE (>90% one pole) — consider DROP_VOTERS"
        if ab > 0.95: flag = " << always abstains — consider DROP_VOTERS"
    if flag: DEGENERATE.append(v)
    print(f"\n{v:12s} n={len(df)}  PZ={pz:.0%}  NG={ng:.0%}  ABSTAIN={ab:.0%}{flag}")

print("\n--- pairwise Cohen kappa (clips where both voters cast a real vote) ---")
for i, a in enumerate(ACTIVE):
    for b in ACTIVE[i + 1:]:
        da, db = ALL_VOTES[a], ALL_VOTES[b]
        if da.empty or db.empty: continue
        m = da[da["vote"].isin(["PZ","NG"])].merge(db[db["vote"].isin(["PZ","NG"])], on="id", suffixes=("_a","_b"))
        m = m[m["id"].isin(report_ids)]
        if len(m) >= 10:
            k = cohen_kappa_score(m["vote_a"], m["vote_b"])
            print(f"  {a:12s} vs {b:12s}: kappa={k:+.3f}  (n={len(m)})")
        else:
            print(f"  {a:12s} vs {b:12s}: <10 joint votes")

print("\nGuidance: kappa >0.2 fair, >0.4 moderate. If a voter is DEGENERATE or has kappa≈0 with everyone,")
print("add it to CONFIG['DROP_VOTERS'] (and optionally enable 'xlsr_reserve'), then set MODE='full' and Run all.")
if not DEGENERATE: print("PILOT OK — no degenerate voters detected.")

In [ ]:
PRESETS = {"strict":  [(4, 0)],
           "default": [(3, 1)],
           "lenient": [(3, 1), (2, 0)]}  
RULES = PRESETS[CONFIG["PRESET"]]

wide = manifest[["id","path","session","seg_idx","speaker_role","duration_s","prefilter","pilot"]].copy()
wide = wide.merge(transcripts[["id","transcript","source","filler_only"]], on="id", how="left")
wide = wide.merge(events[["id","events"]], on="id", how="left")
for v in ACTIVE:
    dv = ALL_VOTES[v]
    if dv.empty: continue
    dv = dv.rename(columns={c: f"{c}_{v}" for c in ["p_pz","p_ng","p_abstain","vote","detail"]})
    wide = wide.merge(dv, on="id", how="left")

def decide(row):
    if row["prefilter"] != "pass":
        return row["prefilter"], "", 0, 0
    ev = str(row.get("events") or "")
    filler = str(row.get("filler_only")).strip().lower() == "true"  
    if filler and ("laughter" not in ev and "crying" not in ev):
        return "rule4_filler", "", 0, 0
    votes = [row.get(f"vote_{v}") for v in ACTIVE]
    n_pz = sum(1 for x in votes if x == "PZ"); n_ng = sum(1 for x in votes if x == "NG")
    for lab, n_for, n_against in (("PZ", n_pz, n_ng), ("NG", n_ng, n_pz)):
        if any(n_for >= mf and n_against <= ma for mf, ma in RULES):
            return "labeled", lab, n_pz, n_ng
    return ("low_agreement" if (n_pz + n_ng) else "insufficient_votes"), "", n_pz, n_ng

res = wide.apply(decide, axis=1, result_type="expand")
wide[["outcome","final_label","n_pz","n_ng"]] = res
wide["agreement"] = (wide[["n_pz","n_ng"]].max(axis=1) / (wide["n_pz"] + wide["n_ng"]).replace(0, np.nan)).round(3)
p_cols = [f"p_pz_{v}" for v in ACTIVE if f"p_pz_{v}" in wide.columns]
wide["mean_p_pz"] = wide[p_cols].mean(axis=1, skipna=True).round(4)

scope = wide[wide["pilot"]] if CONFIG["MODE"] == "pilot" else wide
LABELS_CSV = outpath("labels.csv")
scope.to_csv(LABELS_CSV, index=False)
print(f"Wrote {LABELS_CSV}  (MODE={CONFIG['MODE']}, preset={CONFIG['PRESET']})")
print("\nOutcomes:"); print(scope["outcome"].value_counts().to_string())
lab = scope[scope["outcome"] == "labeled"]
if len(lab):
    n_pz_total = int((lab["final_label"] == "PZ").sum()); n_ng_total = int((lab["final_label"] == "NG").sum())
    print(f"\nLabeled: {len(lab)} clips -> PZ {n_pz_total} ({n_pz_total/len(lab):.0%}) | NG {n_ng_total} ({n_ng_total/len(lab):.0%})")
    print(f"Coverage: {len(lab)}/{len(scope)} = {len(lab)/len(scope):.0%} (the rest is EXCLUDE — Rule 8 favors precision)")

In [ ]:
# statistics, QC exports
def fleiss_kappa(mat):
    """mat: n_items x n_categories counts, equal raters per item."""
    n = mat.sum(axis=1)[0]
    if not (mat.sum(axis=1) == n).all() or n < 2: return float("nan")
    p_cat = mat.sum(axis=0) / mat.sum()
    P_i = ((mat ** 2).sum(axis=1) - n) / (n * (n - 1))
    Pbar, Pe = P_i.mean(), (p_cat ** 2).sum()
    return float((Pbar - Pe) / (1 - Pe)) if Pe < 1 else float("nan")

rep = []
rep.append(f"MODE={CONFIG['MODE']} preset={CONFIG['PRESET']} voters={ACTIVE}")
rep.append(scope["outcome"].value_counts().to_string())

vote_cols = [f"vote_{v}" for v in ACTIVE if f"vote_{v}" in scope.columns]
complete = scope[scope["prefilter"] == "pass"].dropna(subset=vote_cols)
if len(complete):
    cats = ["PZ", "NG", "ABSTAIN"]
    mat = np.stack([[sum(1 for c in vote_cols if row[c] == cat) for cat in cats]
                    for _, row in complete.iterrows()])
    rep.append(f"\nFleiss kappa (all {len(vote_cols)} voters, PZ/NG/ABSTAIN, n={len(complete)}): {fleiss_kappa(mat):.3f}")
    conf = complete[(complete[["n_pz","n_ng"]].sum(axis=1) >= 3)]
    if len(conf):
        # participant counts differ per clip here, so report simple percent agreement
        agree = (conf[["n_pz","n_ng"]].max(axis=1) / conf[["n_pz","n_ng"]].sum(axis=1)).mean()
        rep.append(f"Mean percent agreement on clips with >=3 real votes (n={len(conf)}): {agree:.1%}")

lab = scope[scope["outcome"] == "labeled"].copy()
if len(lab):
    ratio = (lab["final_label"] == "PZ").mean()
    band = "WITHIN" if 0.40 <= ratio <= 0.60 else "OUTSIDE"
    rep.append(f"\nPZ share among labeled: {ratio:.1%} -> {band} the 40/60-60/40 band (Manual §6)")
    if band == "OUTSIDE":
        rep.append("  Remedy order: (1) revisit CONFIG thresholds / PRESET and re-run Cells 15-16 (CPU-only),")
        rep.append("  (2) balance at training time (augment minority / undersample majority).")
    # per-speaker breakdown
    for role, g in lab.groupby("speaker_role"):
        rep.append(f"  {role}: n={len(g)}, PZ share {(g['final_label']=='PZ').mean():.1%}")

    # QC exports 
    qc_cols = ["id","path","transcript","events","final_label","agreement","n_pz","n_ng","mean_p_pz"]
    qc10 = lab.groupby("final_label", group_keys=False).apply(
        lambda g: g.sample(max(1, int(round(len(g) * 0.10))), random_state=CONFIG["SEED"]))
    qc10 = qc10[qc_cols].assign(human_label="")
    qc10.to_csv(outpath("qc_random10.csv"), index=False)
    lowconf = lab[lab["agreement"] < 1.0][qc_cols].assign(human_label="")
    lowconf.to_csv(outpath("qc_low_confidence.csv"), index=False)
    rep.append(f"\nQC exports: qc_random10.csv ({len(qc10)}) and qc_low_confidence.csv ({len(lowconf)})")
    rep.append("            (kept for spot-checks / future human review if ever possible).")
    rep.append("No human labels available -> run the VALIDATION cell (next) for the label-free checks.")

report = "\n".join(rep)
open(outpath("stats_report.txt"), "w").write(report)
print(report)
print("\nAll outputs are in:", CONFIG["OUTPUT_DIR"])

In [ ]:
# LABEL-FREE VALIDATION 

from scipy.stats import mannwhitneyu

val = []   # (check, score, n, threshold)
lab = scope[scope["outcome"] == "labeled"].copy()
if not len(lab):
    print("No labeled clips — nothing to validate.")
else:
    # 1) nonverbal events
    ev = lab["events"].fillna("").astype(str)
    laugh, cry = lab[ev.str.contains("laughter")], lab[ev.str.contains("crying")]
    if len(laugh) >= 5: val.append(("laughter clips labeled PZ", (laugh["final_label"] == "PZ").mean(), len(laugh), 0.70))
    if len(cry) >= 5:   val.append(("crying clips labeled NG",   (cry["final_label"] == "NG").mean(),   len(cry),   0.70))

    # 2) manual keyword concordance
    POS_KW = ["hoş","teşekkür","memnun","inanılmaz","harika","muhteşem","gururlu","rahatlamış",
              "yardımsever","faydalı","havalı","güzel","heyecanlı","keyifli"]
    NEG_KW = ["üzgün","saçmalık","berbat","bok","nefret","mahvol","kork","yapamıyor","yapamam",
              "beceremiyor","beceremem","rahatsız","kötü","ölüm","aptal","çirkin","sinirli","depresif"]
    tl = lab["transcript"].fillna("").astype(str).str.lower()
    has_pos = tl.apply(lambda t: any(k in t for k in POS_KW))
    has_neg = tl.apply(lambda t: any(k in t for k in NEG_KW))
    pos_only, neg_only = lab[has_pos & ~has_neg], lab[has_neg & ~has_pos]
    if len(pos_only) >= 10: val.append(("positive-keyword clips labeled PZ", (pos_only["final_label"] == "PZ").mean(), len(pos_only), 0.60))
    if len(neg_only) >= 10: val.append(("negative-keyword clips labeled NG", (neg_only["final_label"] == "NG").mean(), len(neg_only), 0.60))

    # 3) acoustic construct check (subsampled for speed; session-normalized)
    sub = pd.concat([g.sample(min(len(g), 300), random_state=CONFIG["SEED"]) for _, g in lab.groupby("final_label")])
    feats = []
    for r in tqdm(sub.to_dict("records"), desc="acoustic-check"):
        try:
            y, _ = load_16k(r["id"])
            f0 = librosa.yin(y, fmin=70, fmax=600, sr=16000)
            f0 = f0[np.isfinite(f0)]
            feats.append({"session": r["session"], "final_label": r["final_label"],
                          "f0": float(np.median(f0)) if len(f0) else np.nan,
                          "rms": float(np.sqrt(np.mean(y ** 2)))})
        except Exception:
            pass
    fdf = pd.DataFrame(feats).dropna()
    # within-session normalization needs >=2 clips per session (a lone clip has no std -> NaN)
    fdf = fdf[fdf.groupby("session")["session"].transform("size") >= 2]
    if len(fdf) >= 60:
        for col in ("f0", "rms"):
            fdf[col + "_z"] = fdf.groupby("session")[col].transform(
                lambda x: (x - x.mean()) / (x.std(ddof=0) + 1e-9))
        fdf = fdf.dropna(subset=["f0_z", "rms_z"])
        pz, ng = fdf[fdf["final_label"] == "PZ"], fdf[fdf["final_label"] == "NG"]
        if len(pz) >= 20 and len(ng) >= 20:
            for col, name in (("f0_z", "pitch(F0) PZ>NG"), ("rms_z", "energy(RMS) PZ>NG")):
                u, p = mannwhitneyu(pz[col], ng[col], alternative="greater")
                val.append((f"{name} (Mann-Whitney p={p:.3g})", 1.0 if p < 0.05 else 0.0, len(fdf), 0.99))

    # 4) stability under panel perturbation (leave-one-voter-out, same rules)
    def relabel(rules, active):
        def d(row):
            if row["prefilter"] != "pass": return ""
            e = str(row.get("events") or "")
            filler = str(row.get("filler_only")).strip().lower() == "true"
            if filler and ("laughter" not in e and "crying" not in e): return ""
            n_pz = sum(1 for a in active if row.get(f"vote_{a}") == "PZ")
            n_ng = sum(1 for a in active if row.get(f"vote_{a}") == "NG")
            for l, f, ag in (("PZ", n_pz, n_ng), ("NG", n_ng, n_pz)):
                if any(f >= mf and ag <= ma for mf, ma in rules): return l
            return ""
        return scope.apply(d, axis=1)


    base = scope["final_label"].where(scope["outcome"] == "labeled", "")
    n_base = int((base != "").sum())
    for v_out in ACTIVE:
        alt = relabel(RULES, [a for a in ACTIVE if a != v_out])
        both = (base != "") & (alt != "")
        n_both = int(both.sum())
        if n_both >= 20:
            keep = 1.0 - float(((base != alt) & both).sum()) / n_both
            retention = n_both / max(n_base, 1)
            val.append((f"no flips w/o {v_out} (coverage retention {retention:.0%})", keep, n_both, 0.95))

    # verdicts
    lines = ["================ VALIDATION REPORT (label-free) ================", ""]
    n_pass = 0
    for name, score, n, thr in val:
        ok = score >= thr
        n_pass += ok
        lines.append(f"{'PASS' if ok else 'WARN'}  {name}: {score:.1%} (n={n}, needs >={thr:.0%})")
    lines += ["",
        f"{n_pass}/{len(val)} checks passed."]
    report = "\n".join(lines)
    open(outpath("validation_report.txt"), "w").write(report)
    print(report)